<a href="https://colab.research.google.com/github/7235SYXD/Real-Estate/blob/main/DSP_on_Real_Estate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — INSTALL LIBRARIES                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess, sys
for lib in ["kagglehub","catboost","shap","optuna",
            "vaderSentiment","yake","textstat"]:
    subprocess.run([sys.executable,"-m","pip","install",lib,"-q"])
print("All libraries installed.")


All libraries installed.


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — GOOGLE DRIVE                                           ║
# ╚══════════════════════════════════════════════════════════════════╝

import os, shutil
from google.colab import drive, files

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/RealEstate_TXNY"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

Mounted at /content/drive
Save directory: /content/drive/MyDrive/RealEstate_TXNY


In [3]:
# ╔════════════════════════════════════════════════════╗
# ║  CELL 3 — IMPORT ALL LIBRARIES                      ║
# ╚═════════════════════════════════════════════════════╝

import os
import re
import warnings
from pathlib import Path
import gc # Import the garbage collection module

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn
from sklearn.model_selection import train_test_split, KFold, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, f1_score, classification_report,
    accuracy_score,
)

# Gradient boosting
!pip install catboost
from catboost import CatBoostRegressor, CatBoostClassifier
import xgboost as xgb

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Hyperparameter tuning
!pip install optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Interpretability
import shap

# NLP
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
!pip install yake
import yake
!pip install textstat
import textstat

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 60)
print("All libraries imported successfully.")
print(f"  pandas     : {pd.__version__}")
print(f"  numpy      : {np.__version__}")
print(f"  tensorflow : {tf.__version__}")
print(f"  sklearn    : OK")
print("=" * 60)

All libraries imported successfully.
  pandas     : 2.2.2
  numpy      : 2.0.2
  tensorflow : 2.20.0
  sklearn    : OK


In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — PII COLUMN REGISTRY + STATE MAP                       ║
# ╚══════════════════════════════════════════════════════════════════╝
"""
NEWFIX 2: Define all Personally Identifiable Information (PII) columns
to be dropped from each dataset before any processing begins.

PII columns identified across all 4 datasets:
  SAKIB      : brokered_by (agent/broker name), street (property address),
               prev_sold_date (can identify owner transaction history)
  POLARTECH  : property_url, property_id, broker_id, agent_name,
               agency_name, address, street_name, apartment
  TEXAS 2026 : address, street (if present), agent (if present)
  NEW YORK   : address, street (if present)

Non-PII location columns kept: city, state, zip_code, latitude, longitude
These are geographic but not individually identifying.
"""

# PII columns per dataset (lowercase column names after normalisation)
PII_COLS_SAKIB = [
    "brokered_by",     # agent / broker name — PII
    "street",          # full street address — PII
    "prev_sold_date",  # transaction history — quasi-PII
]

PII_COLS_POLARTECH = [
    "property_url",    # URL can identify listing owner — PII
    "property_id",     # internal ID — quasi-PII
    "broker_id",       # broker identifier — PII
    "agent_name",      # personal name — PII
    "agency_name",     # business name — quasi-PII
    "address",         # full street address — PII
    "street_name",     # partial address — PII
    "apartment",       # unit number — PII
    "agent_phone",     # agent phone number - PII
]

PII_COLS_NLP = [
    "address",         # full address if present — PII
    "street",          # street name — PII
    "agent",           # agent name — PII
    "listing_agent",   # agent name — PII
    "mls_id",          # MLS listing ID — quasi-PII
    "listing_id",      # listing identifier — quasi-PII
]

def drop_pii_columns(df, pii_list, dataset_name):
    """
    Drop PII columns from a DataFrame.
    Reports which columns were found and dropped.
    Only drops columns that actually exist — no error on missing.
    """
    found = [c for c in pii_list if c in df.columns]
    not_found = [c for c in pii_list if c not in df.columns]
    if found:
        df = df.drop(columns=found)
        print(f"  {dataset_name} — PII dropped: {found}")
    if not_found:
        print(f"  {dataset_name} — PII not present (safe): {not_found}")
    return df

# ── STATE MAP ────────────────────────────────────────────────────────────────
STATE_MAP = {
    "ALABAMA":"AL","ALASKA":"AK","ARIZONA":"AZ","ARKANSAS":"AR",
    "CALIFORNIA":"CA","COLORADO":"CO","CONNECTICUT":"CT","DELAWARE":"DE",
    "FLORIDA":"FL","GEORGIA":"GA","HAWAII":"HI","IDAHO":"ID",
    "ILLINOIS":"IL","INDIANA":"IN","IOWA":"IA","KANSAS":"KS",
    "KENTUCKY":"KY","LOUISIANA":"LA","MAINE":"ME","MARYLAND":"MD",
    "MASSACHUSETTS":"MA","MICHIGAN":"MI","MINNESOTA":"MN","MISSISSIPPI":"MS",
    "MISSOURI":"MO","MONTANA":"MT","NEBRASKA":"NE","NEVADA":"NV",
    "NEW HAMPSHIRE":"NH","NEW JERSEY":"NJ","NEW MEXICO":"NM","NEW YORK":"NY",
    "NORTH CAROLINA":"NC","NORTH DAKOTA":"ND","OHIO":"OH","OKLAHOMA":"OK",
    "OREGON":"OR","PENNSYLVANIA":"PA","RHODE ISLAND":"RI","SOUTH CAROLINA":"SC",
    "SOUTH DAKOTA":"SD","TENNESSEE":"TN","TEXAS":"TX","UTAH":"UT",
    "VERMONT":"VT","VIRGINIA":"VA","WASHINGTON":"WA","WEST VIRGINIA":"WV",
    "WISCONSIN":"WI","WYOMING":"WY","DISTRICT OF COLUMBIA":"DC",
}
for abbr in ["AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID",
             "IL","IN","IA","KS","KY","LA","ME","MD","MA","MI","MN","MS",
             "MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK",
             "OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV",
             "WI","WY","DC"]:
    STATE_MAP[abbr] = abbr

def normalise_state(series):
    """Convert any state format to 2-letter abbreviation."""
    return (series.astype(str).str.strip().str.upper()
            .map(lambda x: STATE_MAP.get(x, x)))

# Verify STATE_MAP
test_states = pd.Series(["Texas", "New York", "CA", "florida"])
for raw, norm in zip(test_states, normalise_state(test_states)):
    print(f"  '{raw}' → '{norm}'")
print("State normalisation confirmed ")
print("PII column registry defined ")


  'Texas' → 'TX'
  'New York' → 'NY'
  'CA' → 'CA'
  'florida' → 'FL'
State normalisation confirmed 
PII column registry defined 


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — DOWNLOAD ALL 4 DATASETS                                ║
# ╚══════════════════════════════════════════════════════════════════╝

print("Downloading 4 datasets ...")
path_sakib    = kagglehub.dataset_download("ahmedshahriarsakib/usa-real-estate-dataset")
path_polartech= kagglehub.dataset_download("polartech/500000-us-homes-data-for-sale-properties")
path_texas    = kagglehub.dataset_download("jahnavikachhia23/texas-residential-real-estate-intelligence-2026")
path_newyork  = kagglehub.dataset_download("kanchana1990/new-york-real-estate-data-2026")
print(f"  SAKIB     : {path_sakib}")
print(f"  POLARTECH : {path_polartech}")
print(f"  TEXAS     : {path_texas}")
print(f"  NEW YORK  : {path_newyork}")

100%|██████████| 38.2M/38.2M [00:00<00:00, 46.3MB/s]

Extracting files...


100%|██████████| 34.6M/34.6M [00:00<00:00, 75.4MB/s]

Extracting files...


100%|██████████| 3.46M/3.46M [00:00<00:00, 152MB/s]

Extracting files...


100%|██████████| 3.11M/3.11M [00:00<00:00, 122MB/s]

Extracting files...
  SAKIB     : /root/.cache/kagglehub/datasets/ahmedshahriarsakib/usa-real-estate-dataset/versions/25
  POLARTECH : /root/.cache/kagglehub/datasets/polartech/500000-us-homes-data-for-sale-properties/versions/1
  TEXAS     : /root/.cache/kagglehub/datasets/jahnavikachhia23/texas-residential-real-estate-intelligence-2026/versions/1
  NEW YORK  : /root/.cache/kagglehub/datasets/kanchana1990/new-york-real-estate-data-2026/versions/1


In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — LOAD CSV FILES                                        ║
# ╚══════════════════════════════════════════════════════════════════╝

def find_csv(folder):
    for root, dirs, fs in os.walk(folder):
        for f in fs:
            if f.endswith(".csv"):
                return os.path.join(root, f)
    raise FileNotFoundError(f"No CSV in: {folder}")

df_sakib     = pd.read_csv(find_csv(path_sakib),     low_memory=False)
df_polartech = pd.read_csv(find_csv(path_polartech), low_memory=False)
df_texas     = pd.read_csv(find_csv(path_texas),     low_memory=False)
df_newyork   = pd.read_csv(find_csv(path_newyork),   low_memory=False)

for name, df in [("SAKIB",df_sakib),("POLARTECH",df_polartech),
                  ("TEXAS",df_texas),("NEW YORK",df_newyork)]:
    print(f"{name:10s}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    print(f"  Columns: {list(df.columns)}")

TEXAS_DESC   = "description"
NEWYORK_DESC = "description"

SAKIB     : 2,226,382 rows x 12 cols
  Columns: ['brokered_by', 'status', 'price', 'bed', 'bath', 'acre_lot', 'street', 'city', 'state', 'zip_code', 'house_size', 'prev_sold_date']
POLARTECH : 600,000 rows x 28 cols
  Columns: ['property_url', 'property_id', 'address', 'street_name', 'apartment', 'city', 'state', 'latitude', 'longitude', 'postcode', 'price', 'bedroom_number', 'bathroom_number', 'price_per_unit', 'living_space', 'land_space', 'land_space_unit', 'broker_id', 'property_type', 'property_status', 'year_build', 'total_num_units', 'listing_age', 'RunDate', 'agency_name', 'agent_name', 'agent_phone', 'is_owned_by_zillow']
TEXAS     : 12,137 rows x 13 cols
  Columns: ['type', 'sub_type', 'text', 'listPrice', 'sqft', 'stories', 'beds', 'baths', 'baths_full', 'baths_full_calc', 'garage', 'year_built', 'Price_Per_SqFt']
NEW YORK  : 8,273 rows x 11 cols
  Columns: ['type', 'sub_type', 'text', 'listPrice', 'sqft', 'stories', 'beds', 'baths', 'baths_full', 'baths_full_calc', 'garage'

In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — FULL CLEANING AUDIT: SAKIB                            ║
# ╚══════════════════════════════════════════════════════════════════╝
# Shows: dataset size → duplicates → nulls → outliers → skew →
#        MICE imputation → cleaned size at every step

print("=" * 65)
print("CLEANING AUDIT — SAKIB")
print("=" * 65)

df_sakib_clean = df_sakib.copy()
df_sakib_clean.columns = (df_sakib_clean.columns.str.strip().str.lower()
                           .str.replace(r"[\s\-]+","_",regex=True))

BED_COL  = "bed"        if "bed"        in df_sakib_clean.columns else "beds"
BATH_COL = "bath"       if "bath"       in df_sakib_clean.columns else "baths"
SQFT_COL = "house_size" if "house_size" in df_sakib_clean.columns else "sqfoot"
CITY_COL = "city"       if "city"       in df_sakib_clean.columns else None
print(f"Column mapping: bed='{BED_COL}' bath='{BATH_COL}' sqft='{SQFT_COL}' city='{CITY_COL}'")

# STEP 1: Dataset size
print(f"\nSTEP 1 — Dataset Size")
print(f"  Rows     : {df_sakib_clean.shape[0]:,}")
print(f"  Columns  : {df_sakib_clean.shape[1]}")
print(f"  Memory   : {df_sakib_clean.memory_usage(deep=True).sum()/1e6:.1f} MB")

# STEP 2: PII removal
print("\nSTEP 2 — PII REMOVAL:")
df_sakib_clean = drop_pii_columns(df_sakib_clean, PII_COLS_SAKIB, "SAKIB")

# STEP 3: Duplicate rows
n_before = len(df_sakib_clean)
dup_count = df_sakib_clean.duplicated().sum()
print(f"\nSTEP 3 — Duplicate Rows")
print(f"  Exact duplicates found : {dup_count:,}")
df_sakib_clean = df_sakib_clean.drop_duplicates()
print(f"  Rows after removal     : {len(df_sakib_clean):,} (removed {n_before-len(df_sakib_clean):,})")

# STEP 4: Null values
print(f"\nSTEP 4 — Null Values (before imputation)")
null_counts = df_sakib_clean.isnull().sum()
null_pct    = (null_counts / len(df_sakib_clean) * 100).round(2)
null_report = pd.DataFrame({"null_count":null_counts,"null_pct":null_pct})
null_report = null_report[null_report["null_count"]>0].sort_values("null_count",ascending=False)
print(null_report.to_string())
print(f"\n  Total null values: {df_sakib_clean.isnull().sum().sum():,}")

# Drop rows where price or state is null (cannot impute the target or key ID)
n = len(df_sakib_clean)
df_sakib_clean = df_sakib_clean.dropna(subset=["price","state"])
print(f"  Dropped rows with null price/state: {n-len(df_sakib_clean):,}")

# STEP 5: STATE_MAP fix
print(f"\nSTEP 5 — State Name Normalisation (Bug Fix)")
print(f"  Raw state sample (before): {dict(list(df_sakib_clean['state'].value_counts().head(5).items()))}")
df_sakib_clean["state"] = normalise_state(df_sakib_clean["state"])
print(f"  Normalised sample (after): {dict(list(df_sakib_clean['state'].value_counts().head(5).items()))}")
print(f"  TX rows: {(df_sakib_clean['state']=='TX').sum():,}  NY rows: {(df_sakib_clean['state']=='NY').sum():,}")

# STEP 6: Outlier detection using IQR fencing
print(f"\nSTEP 6 — Outlier Detection and Removal (IQR method)")
print(f"  Price statistics BEFORE clipping:")
print(f"    Min={df_sakib_clean['price'].min():,.0f}  Max={df_sakib_clean['price'].max():,.0f}"
      f"  Mean={df_sakib_clean['price'].mean():,.0f}  Median={df_sakib_clean['price'].median():,.0f}")
q1 = df_sakib_clean["price"].quantile(0.25)
q3 = df_sakib_clean["price"].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 3.0*iqr
upper_fence = q3 + 3.0*iqr
print(f"  IQR = {iqr:,.0f}  Lower fence = ${lower_fence:,.0f}  Upper fence = ${upper_fence:,.0f}")
n = len(df_sakib_clean)
df_sakib_clean = df_sakib_clean[
    (df_sakib_clean["price"] >= lower_fence) &
    (df_sakib_clean["price"] <= upper_fence)]
print(f"  Outliers removed: {n-len(df_sakib_clean):,} rows")
print(f"  Rows remaining  : {len(df_sakib_clean):,}")
print(f"  Price statistics AFTER clipping:")
print(f"    Min={df_sakib_clean['price'].min():,.0f}  Max={df_sakib_clean['price'].max():,.0f}"
      f"  Mean={df_sakib_clean['price'].mean():,.0f}  Median={df_sakib_clean['price'].median():,.0f}")

# STEP 7: Price skewness and log transformation
print(f"\nSTEP 7 — Price Skewness and Log Transformation")
skew_before = df_sakib_clean["price"].skew()
df_sakib_clean["log_price"] = np.log1p(df_sakib_clean["price"])
skew_after  = df_sakib_clean["log_price"].skew()
print(f"  Skewness before log transform: {skew_before:.4f}")
print(f"  Skewness after  log transform: {skew_after:.4f}")


# STEP 8: MICE imputation on structural columns
print(f"\nSTEP 8 — MICE Imputation (Multiple Imputation by Chained Equations)")
safe_cols = [c for c in [BED_COL,BATH_COL,SQFT_COL,"acre_lot"]
             if c in df_sakib_clean.columns]
print(f"  Columns imputed: {safe_cols}")
null_before = {c: df_sakib_clean[c].isnull().sum() for c in safe_cols}
print(f"  Null counts before MICE: {null_before}")
print(f"  Why MICE: Imputes each column based on all other columns iteratively")
print(f"            Better than mean-fill which ignores feature relationships")
mice = IterativeImputer(max_iter=10, random_state=SEED)
df_sakib_clean[safe_cols] = mice.fit_transform(df_sakib_clean[safe_cols])
null_after = {c: df_sakib_clean[c].isnull().sum() for c in safe_cols}
print(f"  Null counts after  MICE: {null_after}")

# STEP 9: Range clipping
print(f"\nSTEP 9 — Range Clipping (beds 0-20, baths 0-20)")
print(f"  Before: bed max={df_sakib_clean[BED_COL].max():.1f}  bath max={df_sakib_clean[BATH_COL].max():.1f}")
df_sakib_clean[BED_COL]  = df_sakib_clean[BED_COL].clip(0,20)
df_sakib_clean[BATH_COL] = df_sakib_clean[BATH_COL].clip(0,20)
print(f"  After : bed max={df_sakib_clean[BED_COL].max():.1f}  bath max={df_sakib_clean[BATH_COL].max():.1f}")


# STEP 10: City normalisation
if CITY_COL:
    df_sakib_clean[CITY_COL] = df_sakib_clean[CITY_COL].astype(str).str.strip().str.upper()
    print(f"\nSTEP 10 — City Column Normalised to Uppercase")
df_sakib_clean["source"] = "sakib"

# Final audit summary
print(f"\nSAKIB CLEANING SUMMARY:")
print(f"  {'Step':<40s}  Rows")
print(f"  {'Original':40s}  {df_sakib.shape[0]:,}")
print(f"  {'After deduplication':40s}  {df_sakib.shape[0]-dup_count:,}")
print(f"  {'After null price/state drop':40s}  {len(df_sakib_clean)+len(df_sakib_clean)-len(df_sakib_clean):,}")
print(f"  {'After IQR outlier removal':40s}  {len(df_sakib_clean):,}")
print(f"  TX={( df_sakib_clean['state']=='TX').sum():,}  NY={(df_sakib_clean['state']=='NY').sum():,}")
assert (df_sakib_clean["state"]=="TX").sum()>0, "TX still 0!"
assert (df_sakib_clean["state"]=="NY").sum()>0, "NY still 0!"
print("SAKIB cleaning complete")

CLEANING AUDIT — SAKIB
Column mapping: bed='bed' bath='bath' sqft='house_size' city='city'

STEP 1 — Dataset Size
  Rows     : 2,226,382
  Columns  : 12
  Memory   : 634.7 MB

STEP 2 — PII REMOVAL:
  SAKIB — PII dropped: ['brokered_by', 'street', 'prev_sold_date']

STEP 3 — Duplicate Rows
  Exact duplicates found : 78,726
  Rows after removal     : 2,147,656 (removed 78,726)

STEP 4 — Null Values (before imputation)
            null_count  null_pct
house_size      511539   23.8200
bath            457363   21.3000
bed             427502   19.9100
acre_lot        310212   14.4400
price             1453    0.0700
city              1304    0.0600
zip_code           297    0.0100
state                8    0.0000

  Total null values: 1,709,678
  Dropped rows with null price/state: 1,461

STEP 5 — State Name Normalisation (Bug Fix)
  Raw state sample (before): {'Florida': 232042, 'California': 224659, 'Texas': 201113, 'New York': 101053, 'Illinois': 83002}
  Normalised sample (after): {'FL':

In [8]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — FULL CLEANING AUDIT: POLARTECH                        ║
# ╚══════════════════════════════════════════════════════════════════╝

print("=" * 65)
print("CLEANING AUDIT — POLARTECH")
print("=" * 65)

df_polartech_clean = df_polartech.copy()
df_polartech_clean.columns = (df_polartech_clean.columns.str.strip().str.lower()
                               .str.replace(r"[\s\-]+","_",regex=True))

# STEP 1: Dataset size
print(f"\nSTEP 1 — Dataset Size")
print(f"  Rows     : {df_polartech_clean.shape[0]:,}")
print(f"  Columns  : {df_polartech_clean.shape[1]}")
print(f"  Memory   : {df_polartech_clean.memory_usage(deep=True).sum()/1e6:.1f} MB")

# PII removal
print("\nSTEP 2 — PII REMOVAL:")
df_polartech_clean = drop_pii_columns(df_polartech_clean, PII_COLS_POLARTECH, "POLARTECH")

# STEP 3: Duplicates
n_before = len(df_polartech_clean)
dup_count_p = df_polartech_clean.duplicated().sum()
print(f"\nSTEP 3 — Duplicate Rows")
print(f"  Exact duplicates: {dup_count_p:,}")
df_polartech_clean = df_polartech_clean.drop_duplicates()
print(f"  Rows after removal: {len(df_polartech_clean):,}")

# STEP 4: Nulls
print(f"\nSTEP 4 — Null Values")
null_p = df_polartech_clean.isnull().sum()
null_pct_p = (null_p/len(df_polartech_clean)*100).round(2)
null_rep_p = pd.DataFrame({"null_count":null_p,"null_pct":null_pct_p})
null_rep_p = null_rep_p[null_rep_p["null_count"]>0].sort_values("null_count",ascending=False)
print(null_rep_p.head(10).to_string())
print(f"  Total nulls: {df_polartech_clean.isnull().sum().sum():,}")

n = len(df_polartech_clean)
df_polartech_clean = df_polartech_clean.dropna(subset=["price","state"])
print(f"  Dropped null price/state: {n-len(df_polartech_clean):,}")

# STEP 5: State normalisation
print(f"\nSTEP 5 — State Name Normalisation")
print(f"  Raw sample: {dict(list(df_polartech_clean['state'].value_counts().head(5).items()))}")
df_polartech_clean["state"] = normalise_state(df_polartech_clean["state"])
print(f"  After fix : TX={( df_polartech_clean['state']=='TX').sum():,}  NY={(df_polartech_clean['state']=='NY').sum():,}")

# STEP 6: Outlier detection
print(f"\nSTEP 6 — Outlier Detection (IQR)")
print(f"  Price before: Min={df_polartech_clean['price'].min():,.0f}  Max={df_polartech_clean['price'].max():,.0f}")
q1p=df_polartech_clean["price"].quantile(0.25); q3p=df_polartech_clean["price"].quantile(0.75)
iqrp=q3p-q1p
n=len(df_polartech_clean)
df_polartech_clean = df_polartech_clean[
    (df_polartech_clean["price"]>=q1p-3.0*iqrp) &
    (df_polartech_clean["price"]<=q3p+3.0*iqrp)]
print(f"  Outliers removed: {n-len(df_polartech_clean):,}  Remaining: {len(df_polartech_clean):,}")

# STEP 7: Log price
print(f"\nSTEP 7 — Log Price Transformation")
skew_p_before = df_polartech_clean["price"].skew()
df_polartech_clean["log_price"] = np.log1p(df_polartech_clean["price"])
skew_p_after  = df_polartech_clean["log_price"].skew()
print(f"  Skewness before: {skew_p_before:.4f}  After: {skew_p_after:.4f}")

# STEP 8: Data type consistency
print(f"\nSTEP 8 — Data Type Standardisation")
if "city" in df_polartech_clean.columns:
    df_polartech_clean["city"] = df_polartech_clean["city"].astype(str).str.strip().str.upper()
    print(f"  City column normalised to uppercase")

# STEP 9: Negative price check
neg_prices = (df_polartech_clean["price"] < 0).sum()
print(f"\nSTEP 9 — Negative/Zero Price Check")
print(f"  Negative prices found: {neg_prices}  {'(none — clean ✓)' if neg_prices==0 else '— removing'}")
if neg_prices > 0:
    df_polartech_clean = df_polartech_clean[df_polartech_clean["price"]>0]

df_polartech_clean["source"] = "polartech"
print(f"\nPOLARTECH CLEANING SUMMARY:")
print(f"  Original : {df_polartech.shape[0]:,}")
print(f"  Final    : {df_polartech_clean.shape[0]:,}")
print(f"  TX={( df_polartech_clean['state']=='TX').sum():,}  NY={(df_polartech_clean['state']=='NY').sum():,}")
print("POLARTECH cleaning complete")

CLEANING AUDIT — POLARTECH

STEP 1 — Dataset Size
  Rows     : 600,000
  Columns  : 28
  Memory   : 542.1 MB

STEP 2 — PII REMOVAL:
  POLARTECH — PII dropped: ['property_url', 'property_id', 'broker_id', 'agent_name', 'agency_name', 'address', 'street_name', 'apartment', 'agent_phone']

STEP 3 — Duplicate Rows
  Exact duplicates: 11,157
  Rows after removal: 588,843

STEP 4 — Null Values
                 null_count  null_pct
year_build           588843  100.0000
total_num_units      588843  100.0000
price_per_unit       157070   26.6700
bedroom_number       148433   25.2100
living_space         144907   24.6100
bathroom_number      121814   20.6900
land_space_unit       82771   14.0600
land_space            82771   14.0600
latitude              65336   11.1000
longitude             65336   11.1000
  Total nulls: 2,046,156
  Dropped null price/state: 1

STEP 5 — State Name Normalisation
  Raw sample: {'TX': 144002, 'CA': 101947, 'IL': 43368, 'AZ': 36909, 'MO': 31186}
  After fix : TX=14